## HFP Hypothesis 1 Test: Importance-Based Retention
Bu hücre, HFP modelinin dış bir sinyal (attention score) olmadan, kritik kelimeleri kalın yazmayı (büyük L2 normu) kendi kendine öğrenip öğrenmediğini sınar. Model küçük bir veri setinde eğitilir ve `W_k` projeksiyonunun ürettiği vektör normları analiz edilir.

In [ ]:
import random
import torch
import numpy as np
import sys

# Repo dizinini ekle (Colab'de '/content/HFP' olabilir)
# sys.path.append('/content/HFP')

from hfp.models.configuration_hfp import HFPConfig
from hfp.models.modeling_hfp import HFPForCausalLM

KLO, KHI = 100, 130
VLO, VHI = 130, 160
FHI = 100
CTX, P, STEPS = 160, 8, 500

def make_seq(ctx, P):
    toks = [random.randint(1, FHI - 1) for _ in range(ctx)]
    keys = random.sample(range(KLO, KHI), P)
    lab = [-100] * ctx
    slots = sorted(random.sample(range(ctx // 2), P))
    for i, slot in enumerate(slots):
        wpos = slot * 2
        k = keys[i]
        v = random.randint(VLO, VHI - 1)
        toks[wpos] = k
        toks[wpos + 1] = v
        qp = ctx - (P - i)*2
        toks[qp] = k
        toks[qp + 1] = 0
        lab[qp + 1] = v
    return toks, lab

def get_batch(bs=16):
    xs, ys = zip(*[make_seq(CTX, P) for _ in range(bs)])
    return torch.tensor(xs), torch.tensor(ys)

print("Hypothesis 1 Test: Importance-Based Retention")
cfg = HFPConfig(vocab_size=VHI + 4, hidden_size=64, num_hidden_layers=1,
                num_attention_heads=2, intermediate_size=256, bulk_dim=32,
                short_len=8, max_position_embeddings=1288, local_window=8,
                decay_mode="cubic_flux", rec_block=32, write_rule="delta", key_feature_map="dpfp")
                
model = HFPForCausalLM(cfg).cuda()
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)

model.train()
for step in range(1, STEPS + 1):
    x, y = get_batch(bs=32)
    out = model(x.cuda(), labels=y.cuda())
    opt.zero_grad()
    out.loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()

model.eval()
captured_k = []
hook = model.hfp.layers[0].hfp_bulk.W_k.register_forward_hook(lambda m, i, o: captured_k.append(o.detach()))

x_test, _ = get_batch(bs=32)
with torch.no_grad():
    model(x_test.cuda())
hook.remove()

k_norms_flat = torch.norm(captured_k[0], p=2, dim=-1).flatten().cpu().numpy()
x_flat = x_test.flatten().cpu().numpy()

key_mask = (x_flat >= KLO) & (x_flat < KHI)
filler_mask = (x_flat >= 1) & (x_flat < FHI)

mean_key = np.mean(k_norms_flat[key_mask])
mean_filler = np.mean(k_norms_flat[filler_mask])

print(f"Key Token L2 Norm: {mean_key:.4f}")
print(f"Filler Token L2 Norm: {mean_filler:.4f}")
print(f"Fark Çarpanı: {mean_key / mean_filler:.2f}x")
if mean_key > mean_filler * 1.5:
    print("SONUÇ: HİPOTEZ DOĞRULANDI.")
else:
    print("SONUÇ: ETKİ ZAYIF VEYA REDDEDİLDİ.")
